In [ ]:
import h5py
import numpy as np
import torch

path = "/path/to/fastmri/knee_singlecoil_val/file000.h5"  # adjust

with h5py.File(path, "r") as f:
    print("Keys:", list(f.keys()))
    kspace = f["kspace"][:]  # shape: [num_slices, Ny, Nx] or [num_slices, coils, Ny, Nx]
    print("kspace shape:", kspace.shape, "dtype:", kspace.dtype)

    if "reconstruction_rss" in f:
        recon = f["reconstruction_rss"][:]   # magnitude images
        print("recon_rss shape:", recon.shape, "dtype:", recon.dtype)


In [ ]:
import glob
from torch.utils.data import Dataset, DataLoader

class FastMRIDataset(Dataset):
    def __init__(self, h5_paths, undersample_mask=None, max_slices=None):
        self.paths = h5_paths
        self.undersample_mask = undersample_mask  # e.g., [Ny,] or [Ny,1]
        self.samples = []  # list of (path, slice_idx)

        for p in self.paths:
            with h5py.File(p, "r") as f:
                num_slices = f["kspace"].shape[0]
            for s in range(num_slices):
                self.samples.append((p, s))
                if max_slices is not None and len(self.samples) >= max_slices:
                    break
            if max_slices is not None and len(self.samples) >= max_slices:
                break

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, slice_idx = self.samples[idx]
        with h5py.File(path, "r") as f:
            kspace = f["kspace"][slice_idx]   # [Ny, Nx], complex64

        kspace = torch.from_numpy(kspace)    # complex64

        # Optionally apply undersampling mask along phase-encode
        if self.undersample_mask is not None:
            # assume mask shape [Ny] or [Ny,1], broadcast over Nx
            mask = torch.from_numpy(self.undersample_mask).to(kspace.device)
            kspace_und = kspace * mask
        else:
            kspace_und = kspace

        # IFFT to get complex image (centered)
        img_full = torch.fft.ifft2(kspace)        # [Ny, Nx], complex64
        img_und  = torch.fft.ifft2(kspace_und)    # [Ny, Nx], complex64

        # Add channel dim for Conv2d/CVConv2d: [1, H, W]
        img_und  = img_und.unsqueeze(0)   # input
        img_full = img_full.unsqueeze(0)  # target

        return img_und, img_full


In [ ]:
train_paths = sorted(glob.glob("/path/to/fastmri/knee_singlecoil_train/*.h5"))
val_paths   = sorted(glob.glob("/path/to/fastmri/knee_singlecoil_val/*.h5"))

train_ds = FastMRIDataset(train_paths, undersample_mask=None, max_slices=2000)
val_ds   = FastMRIDataset(val_paths,   undersample_mask=None, max_slices=500)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False, num_workers=4, pin_memory=True)

x0, y0 = next(iter(train_loader))
print("x0:", x0.shape, x0.dtype)  # expect [B,1,H,W], complex
print("y0:", y0.shape, y0.dtype)


In [ ]:
import torchcvnn.nn as c_nn

def get_c_activation(name):
    name = name.lower()
    if name == "cardioid":
        return c_nn.Cardioid()
    if name == "modrelu":
        return c_nn.modReLU()
    if name == "zrelu":
        return c_nn.zReLU()
    return c_nn.CReLU()  # type A baseline

class ComplexMRIUNetSmall(nn.Module):
    def __init__(self, act_name="cardioid"):
        super().__init__()
        act = get_c_activation(act_name)

        self.enc = nn.Sequential(
            c_nn.ConvTranspose2d(1, 16, kernel_size=3, padding=1),
            c_nn.BatchNorm2d(16),
            act,
            c_nn.ConvTranspose2d(16, 32, kernel_size=3, padding=1),
            c_nn.BatchNorm2d(32),
            act,
            c_nn.AvgPool2d(2),
            c_nn.ConvTranspose2d(32, 64, kernel_size=3, padding=1),
            c_nn.BatchNorm2d(64),
            act,
            c_nn.AvgPool2d(2),
        )

        self.dec = nn.Sequential(
            c_nn.ConvTranspose2d(64, 32, kernel_size=3, padding=1),
            c_nn.BatchNorm2d(32),
            act,
            c_nn.ConvTranspose2d(32, 16, kernel_size=3, padding=1),
            c_nn.BatchNorm2d(16),
            act,
            c_nn.ConvTranspose2d(16, 1, kernel_size=3, padding=1),
        )

    def forward(self, x):
        z = self.enc(x)      # complex
        z = self.dec(z)      # complex
        return z             # complex reconstruction


In [ ]:
def complex_mse(pred, target):
    return ((pred.real - target.real)**2 + (pred.imag - target.imag)**2).mean()


In [ ]:
def train_one_epoch(model, loader, optimizer, epoch, tag="cardioid"):
    model.train()
    total_loss = 0.0
    total = 0

    for x, y in tqdm(loader, desc=f"[{tag}] Train {epoch}", leave=False):
        x = x.to(device)      # complex
        y = y.to(device)      # complex

        optimizer.zero_grad()
        y_hat = model(x)
        loss = complex_mse(y_hat, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        total += x.size(0)

    print(f"[{tag}] Epoch {epoch} | loss={total_loss/total:.6f}")


In [ ]:
def complex_to_two_channels(x):
    # x: [B,1,H,W] complex
    xr = torch.view_as_real(x)        # [B,1,H,W,2]
    xr = xr.squeeze(1).permute(0, 3, 1, 2)  # [B,2,H,W]
    return xr.float()
